In [1]:
import numpy as np
import timeit
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
from numpy.linalg import eig
import torch
import random
from SALib.sample import saltelli
from SALib.analyze import sobol
import multiprocessing as mp
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

In [2]:
np.set_printoptions(precision=6, suppress=True)
torch.set_printoptions(precision=6)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Times New Roman'

In [3]:
problem = {
    'num_vars': 10,
    'names': ['q', 'alpha', 'beta', 'c', 'b', 'kappa', 'k1', 'k2', 'k3', 'tp'],
    'bounds': [[0.5, 0.99],       # Fractional derivative order q
               [-5, 5],           # System parameter alpha
               [-5, 5],           # System parameter beta
               [1, 10],           # c > 0
               [1, 10],           # b > 0
               [0.01, 2],         # kappa > 0
               [1, 10],           # k1 > 0
               [1, 10],           # k2 > 0
               [0.00001, 10],     # k3 > 0
               [0.02, 0.18]]      # tp
}

In [4]:
param_values = saltelli.sample(problem, 256) # Generate 256 combinations of the params

### **Other Global constants ($n$, $t_0$, $t_{final}$, $d$, $p_{d_i}$)**

In [5]:
# n followers
n = 4

# time
t0 = 0.00
tf = 0.20

# time step length
h = 0.001 
num_steps = round((tf-t0)/h)

# Caputo fractional order
q = 0.8

# pinning mechananism indicator (i.e. 1: connection exists; 0: no connection between the agent and the leader)
d = 1

In [6]:
# desired displacement between the i-th follower and the leader
# example: pdi = [[-2,2], [-2,-2], [2,-2], [2,2]]

pdi = {1:-2, 2:2, 3:-2, 4:-2, 5:2, 6:-2, 7:2, 8:2}

### **Global matrices ($A_{adj}$, $D_{diagonal}$, $L$, $\bar{A}$, $\bar{C}$)**

In [7]:
# left inside the main func

## **Function 1: Numerical solver for Caputo fractional-order system**

In [8]:
from scipy.special import gamma
import matplotlib.pyplot as plt

In [9]:
# calculate needed params: p_hat, v_hat, p_bar for the leader_follower dynamics
def phi_calculator(k3, t0, tp, t):
    # under condition sigma(t)==1
    # ϕσ(t)(t) = e^{k3(t0+tp−t)}
    e = np.exp(1)
    if t < tp:
        return (e**(k3*(t0+tp-t)))-1
    elif t >= tp:
        return e**(-t)

def dot_phi_calculator(k3, t0, tp, t):
    e = np.exp(1)
    if t < tp:
        return (-k3)*(e**(k3*(t0+tp-t)))
    elif t >= tp:
        return (-1)*e**(-t)

def p_hat_calculator(pi, p0, pdi_val):
    return pi-p0-pdi_val

def v_hat_calculator(vi, v0):
    return vi-v0

def f(v):
    return 0.05*(v**2)

In [10]:
# virtual leader and virtual follower dynamics step by step (final choice)
# counter added for assigning the pdi_key
def f_leader_follower(X, t, pdi_val, key, q, alpha, beta, c, b, kappa, k1, k2, k3, tp, P, V):
    p0, v0, pi, vi = X

    u_p = (-k2) * c * p_hat_calculator(pi, p0, pdi_val)
    u_pt1 = k3 * (dot_phi_calculator(k3, t0, tp, t) / phi_calculator(k3, t0, tp, t)) * p_hat_calculator(pi, p0, pdi_val)

    # see equation 7
    # Ni: neighbor set for every pi which has connection with others
    # follower_index: the input lines are divided into x and y components
#    follower_index = np.round(int(np.ceil(pdi_key/2)))

#    if Ni[follower_index]==[]: # if the neighbor set of the current pi is empty, Pc=0
#        Pc = 0
#    else:
#        # save the pj_hat and vj_hat into one dictionary? so that they could be retrieved (PVj[j][0]: pj_hat)
#        Pc = np.sum( p_hat_calculator(pi, p0, pdi_val) - PVj[j][0] + v_hat_calculator(vi, v0) - PVj[j][1])

    Pc = 0
    if key == 1:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v4x'][-1], V['v0x'][-1]))
        # Pl = 0
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 2:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v4y'][-1], V['v0y'][-1]))
        # Pl = 0
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 3:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v1x'][-1], V['v0x'][-1]))
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 4:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v1y'][-1], V['v0y'][-1]))
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 5:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v2x'][-1], V['v0x'][-1]))
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 6:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v2y'][-1], V['v0y'][-1]))
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 7:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v3x'][-1], V['v0x'][-1]))
        # Pl = 0
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))
    elif key == 8:
        Pc = (-b)*(v_hat_calculator(vi, v0) - v_hat_calculator(V['v3y'][-1], V['v0y'][-1]))
        # Pl = 0
        Pl = (-c) * d * (p_hat_calculator(pi, p0, pdi_val) + v_hat_calculator(vi, v0))

    u_v = alpha * pdi_val + k1*Pc + k2*Pl
    u_pt2 = k3 * (dot_phi_calculator(k3, t0, tp, t) / phi_calculator(k3, t0, tp, t)) * v_hat_calculator(vi, v0)


    # ================================================================================
    # ================================================================================

    # get the dynamics for one specific set of states [p0, v0, pi, vi]

    val = np.array([[v0,
                     alpha*p0 + beta*v0 + f(v0),
                     vi + u_p + u_pt1,
                     alpha*pi + beta*vi + f(vi) + u_v + u_pt2]])

    # =================================================================================
    # =================================================================================

    return val

In [11]:
def f_abm_solver(f_uav, initial_states, pdi, q, alpha, beta, c, b, kappa, k1, k2, k3, tp, t0, h, num_steps, current_step, key, P, V):
    # initialize arrays to store the solution and time points
    num_equations = len(initial_states)
    solution = np.zeros((num_steps+1, num_equations))

    # set initial conditions
    solution[0] = initial_states

    # Euler's method for 2 very first solutions
    t = round(t0+h, 5)
    f = f_uav(solution[0], t, pdi, key, q, alpha, beta, c, b, kappa, k1, k2, k3, tp, P, V)
    h_q = (h**q) / gamma(q+1)
    solution[1] = solution[0] + h_q*f

    # Fractional Adams-Bashforth iteration
    for j in range(0, num_steps):
        t = round(t0+j*h, 5)

        # first-order FABM
        sum_prev = 0
        for k in range(1, 0+1):
            sum_prev += (solution[j-k] - solution[j-k-1]) / (h**q)

        f = f_uav(solution[j], t, pdi, key, q, alpha, beta, c, b, kappa, k1, k2, k3, tp, P, V)
        solution[j+1] = solution[j] + (h**q)/gamma(q+1) * (f+sum_prev)

    return solution[current_step] # the current step solution

## **Function 2: Caputo Fractional Derivative Calculator**

In [12]:
import control as ct

# need a calculator which can work with tensors
from pyfod.fod import caputo
from scipy.interpolate import CubicSpline

In [13]:
from scipy.special import gamma

In [14]:
# calculate parameter: aa
# input the subscript l and the Caputo fraction order q
def aa(l, q):
    val = ((l+1)**(1-q)) - (l**(1-q))
    return val

# calculate parameter: bb
def bb(l, q):
    val = (((l+1)**(2-q)-l**(2-q))/(2-q)) - (((l+1)**(1-q)+l**(1-q))/2)
    return val

# calculate parameter: cc
def cc(l, q):
    val = 1/6 * (((l+1)**(1-q))+2*(l**(1-q))) + (1/(2-q))*(l**(2-q)) - ((((l+1)**(3-q))-(l**(3-q)))/((2-q)*(3-q)))
    return val

# calculate parameter: dd
def dd(j, l, q):
    # j: jth time point
    # l: subscript for dd, smaller than j. (for example in the summation term l = j-k-1)
    # q: the Caputo fractional order
    val = np.zeros(j)

    if j == 1:
        val[0] = 1
    elif j == 2:
        val[0] = aa(0, q) + bb(0, q)
        val[1] = aa(1, q) - bb(0, q)
    elif j == 3:
        val[0] = aa(0, q) + bb(0, q) + cc(0, q)
        val[1] = aa(1, q) + bb(1, q) - bb(0, q) - 2*cc(0, q)
        val[2] = aa(2, q) - bb(1, q) + cc(0, q)
    elif j >= 4:
        val[0] = aa(0, q) + bb(0, q) + cc(0, q)
        val[1] = aa(1, q) + bb(1, q) - bb(0, q) + cc(1, q) - 2*cc(0, q)
        # l = [2, j-3]
        for i in range(2, j-2):
            val[i] = aa(i, q) + bb(i, q) - bb(i-1, q) + cc(i, q) - 2*cc(i-1, q) + cc(i-2, q)
        val[j-2] = aa(j-2, q) + bb(j-2, q) - bb(j-3, q) - 2*cc(j-3, q) + cc(j-4, q)
        val[j-1] = aa(j-1, q) - bb(j-2, q) + cc(j-3, q)

    return val[l]

In [15]:
def Caputo_frac_derivative_calculator(t, t0, h, H_val, q, alpha, beta, c, b, kappa, k1, k2, k3, tp):
    # t: the current time t, used in all same-time calculations (H(t), R(t), phi(t)...) at that time point
    # h: time step h = (tp-t0)/num_steps
    # V_val: length = j, until the jth time point t
    # q: the Caputo fractional order, pre-defined

    # output a single value dH, Caputo fractional order derivative of H(t)
    # jth time point t stating from t0
    j = round((t-(t0))/h)

    summation = 0
    if j == 1:
        # summation term doesn't exist, k cannot go from 1 to 0
        summation = 0
    else:
        # calculate the summation term
        # summation index from 1 to j-1
        for k in range(1,j):
            dd_jk = dd(j, j-k, q) # param dd inside the summation with subscript j-k
            dd_jk1 = dd(j, j-k-1, q) # param dd inside the summation with subscript j-k-1
            summation = summation + ((dd_jk1-dd_jk)*H_val[k])

    dd_0 = dd(j, 0, q)
    dd_j1 = dd(j, j-1, q)

    dV = ((h**(-q))/gamma(2-q)) * (dd_0*H_val[-1] - summation - dd_j1*H_val[0])

    return dV # return just the single dh at that time point

## **Function 3: Fractional Integral Calculator**

In [16]:
# computes the fractional integral of f(t) from t_a to t_a+(Delta_t*num_steps) of order q
def frac_integral(f_list, t_a, t_b, q, Delta_t):
    # f(t): function to integrate
    # t_a: lower limit of integration

    # initialize fractional integral
    If_val = 0.0

    # number of steps
    integ_steps = round((t_b-t_a)/Delta_t)

    # time steps
    t_values = np.linspace(t_a, t_b, integ_steps+1)

    # numerical integration using trapezoidal rule
    for i in range(1, integ_steps+1):  # include t_b boundary
        tau = t_values[i-1]
        tau_next_step = t_values[i]

        # avoid dividing by 0 at tau = t_b
        if i == integ_steps:
            If_val = If_val + ((t_b-tau)**(q-1)*(f_list[i-1])) * Delta_t / 2

        else:
            # trapezoidal rule: average of two consecutive points; integrand: (-f_list(tau)) / (t_b-tau)^(alpha)
            If_val = If_val + ((t_b-tau)**(q-1)*(f_list[i-1]) + (t_b-tau_next_step)**(q-1)*(f_list[i])) * Delta_t / 2

    If_val = If_val / gamma(1+q)

    return If_val

## **PTs_NN for learning suitable set of matrix elements (to make (15) valid)**

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [18]:
class PTs_Net(nn.Module):
        
    # if you want to use the existing weight matrices then uncomment this + comment the __init__() func behind
    def __init__(self, P_w, n_features, epsilon=1e-5):
        super(PTs_Net, self).__init__()
        self.n = n_features
        self.epsilon = epsilon
        self.weight = nn.Parameter(torch.tensor(P_w, dtype=torch.float), requires_grad=True)
        n_params = n_features * (n_features + 1) // 2
        self.L_params = nn.Parameter(torch.randn(n_params) * 0.1)

    '''
    def __init__(self, n_features, epsilon=1e-6):
        super().__init__()
        self.n = n_features
        self.epsilon = epsilon
        n_params = n_features * (n_features + 1) // 2
        self.L_params = nn.Parameter(torch.randn(n_params) * 0.1)
    '''
    
    def forward(self, in_X):
        if not torch.is_tensor(in_X):
            in_X = torch.tensor(in_X, dtype=torch.float)
            
        L = torch.zeros(self.n, self.n, dtype=torch.float)
        idx = 0
        for i in range(self.n):
            for j in range(i+1):
                L[i, j] = self.L_params[idx]
                idx += 1
        # positive definite P = L L^T + εI
        P = L @ L.T + self.epsilon * torch.eye(self.n)
        # H(t) = 0.5 * X^T P X
        H = 0.5 * (in_X @ P @ in_X.T).squeeze()
        return H, P

In [19]:
n_features = 16
learning_rate = 0.0005

#### **Always Positive Definite**

In [20]:
P_w = np.loadtxt('/Users/ekeulseuji/Downloads/FinalP.txt')
PTs_model = PTs_Net(P_w, n_features)

optimizer = optim.Adam(PTs_model.parameters(), lr=learning_rate, weight_decay=1e-6)

## **Sensitivity Analysis Funcs**

#### For every input set (one sample leader-follower group), calculate $\bar{X}$, the Pts_NN input

In [21]:
zero_tensor = torch.tensor(0, dtype=torch.float)

def run_SA_simulation(single_param_set, P, V):
    P_init = P; V_init = V
    q, alpha, beta, c, b, kappa, k1, k2, k3, tp = single_param_set

    # adjacency matrix A_adj, 1->2, 2->3, 3->4, 4->1
    A_adj = [[0, 0, 1, 0, 0, 0, 0, 0],
             [0, 0, 0, 1, 0, 0, 0, 0],
             [0, 0, 0, 0, 1, 0, 0, 0],
             [0, 0, 0, 0, 0, 1, 0, 0],
             [0, 0, 0, 0, 0, 0, 1, 0],
             [0, 0, 0, 0, 0, 0, 0, 1],
             [1, 0, 0, 0, 0, 0, 0, 0],
             [0, 1, 0, 0, 0, 0, 0, 0]]

    # initialize D_dia to be a zero matrix
    D_dia = [[0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0, 0, 0, 0]]

    # D_dia is a diagonal matrix with l_ii = ( Σ(i≠j) a_ij )
    for i in range(len(A_adj)):
        D_dia[i][i] = sum(A_adj[i][j] for j in range(len(A_adj)) if i!=j)

    # # D_dia is a diagonal matrix with l_ij = −a_ij, for ∀i≠j
    for i in range(len(A_adj)):
        for j in range(len(A_adj)):
            if i!=j:
                D_dia[i][j] = -A_adj[i][j]

    # Laplacian matrix of the system
    L = np.array(D_dia) - np.array(A_adj)

    # A_bar
    A = [0,0,1,0,0,0,0,1,alpha,0,beta,0,0,alpha,0,beta]
    A = np.array(A).reshape(4,4)

    I = np.eye(4)

    A_bar = np.kron(A,I).reshape(16,16)

    ones = [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]
    ones = np.array(ones).reshape(16,1)

    # X_bar = np.kron(X, ones).reshape(16,16)
    # X_bar_T = np.transpose(X_bar)

    # C_bar

    upper_left = -(k2*c)*np.eye(8) # the upper left part of the matric C_bar
    upper_right = np.zeros(64).reshape(8,8) # the lower right part of the matrix C_bar

    C_upper = np.hstack((upper_left, upper_right))

    lower_left = -(k2*c*d)*np.eye(8) - k1*L # the lower left part of the matric C_bar
    lower_right = -(k2*c*d)*np.eye(8) - k1*L # the lower right part of the matrix C_bar

    C_lower = np.hstack((lower_left, lower_right))

    C_bar = np.vstack((C_upper, C_lower))
    # C_bar.shape
    
    model_loss = 1000
    n_iter = 0 # number of learning iterations
    max_iter = 5000
    H_epsilon = 1e-3
    
    while model_loss > 0 and n_iter <= max_iter: 
        H_list = []; U_list = []; R_list = []; X_norm_list = []; t_list = []; q1_list = []
        loss1_sum = zero_tensor; loss2_sum = zero_tensor; lossR_sum = zero_tensor; running_loss = zero_tensor
        losses = []

        P = P_init; V = V_init
        for t in np.arange(t0, tf+h, h):
            t_list.append(t)
            pdi_key = 1
    
            # p_bar = [p1_hat^T, p2_hat^T, p3_hat^T, P4_hat^T]^T
            # p_bar = [p1_hat_x, p1_hat_y, p2_hat_x, p2_hat_y, p3_hat_x, p3_hat_y, p4_hat_x, p4_hat_y]^T
            p_bar = []; v_bar = []
    
            for pv_line in input_pv:
                pdi_value = pdi[pdi_key]
    
                pv_sol = f_abm_solver(f_leader_follower, np.array(pv_line), pdi_value, q, alpha, beta, c, b, kappa, k1, k2, k3, tp, t0, h, num_steps, round((t-t0)/h), pdi_key, P, V)
            
                if pdi_key == 1:
                    csi_hat_norm = 0
                    if n_iter == 0:
                        P['p0x'].append(pv_sol[0]); V['v0x'].append(pv_sol[1]); P['p1x'].append(pv_sol[2]); V['v1x'].append(pv_sol[3])
                    p1_hat_x = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[1]); v1_hat_x = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p1_hat_x**2 + v1_hat_x**2

                elif pdi_key == 2:
                    if n_iter == 0:
                        P['p0y'].append(pv_sol[0]); V['v0y'].append(pv_sol[1]); P['p1y'].append(pv_sol[2]); V['v1y'].append(pv_sol[3])
                    p1_hat_y = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[2]); v1_hat_y = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p1_hat_y**2 + v1_hat_y**2

                elif pdi_key == 3:
                    if n_iter == 0:
                        P['p2x'].append(pv_sol[2]); V['v2x'].append(pv_sol[3])
                    p2_hat_x = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[3]); v2_hat_x = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p2_hat_x**2 + v2_hat_x**2

                elif pdi_key == 4:
                    if n_iter == 0:
                        P['p2y'].append(pv_sol[2]); V['v2y'].append(pv_sol[3])
                    p2_hat_y = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[4]); v2_hat_y = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p2_hat_y**2 + v2_hat_y**2

                elif pdi_key == 5:
                    if n_iter == 0:
                        P['p3x'].append(pv_sol[2]); V['v3x'].append(pv_sol[3])
                    p3_hat_x = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[5]); v3_hat_x = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p3_hat_x**2 + v3_hat_x**2

                elif pdi_key == 6:
                    if n_iter == 0:
                        P['p3y'].append(pv_sol[2]); V['v3y'].append(pv_sol[3])
                    p3_hat_y = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[6]); v3_hat_y = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p3_hat_y**2 + v3_hat_y**2

                elif pdi_key == 7:
                    if n_iter == 0:
                        P['p4x'].append(pv_sol[2]); V['v4x'].append(pv_sol[3])
                    p4_hat_x = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[7]); v4_hat_x = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p4_hat_x**2 + v4_hat_x**2

                elif pdi_key == 8:
                    if n_iter == 0:
                        P['p4y'].append(pv_sol[2]); V['v4y'].append(pv_sol[3])
                    p4_hat_y = p_hat_calculator(pv_sol[2], pv_sol[0], pdi[8]); v4_hat_y = v_hat_calculator(pv_sol[3], pv_sol[1])
                    csi_hat_norm = csi_hat_norm + p4_hat_y**2 + v4_hat_y**2

                pdi_key = pdi_key + 1
    
            # should go to zero as time processes,
            # meaning that the leader-follower system will maintain their pre-specified realtive positions and velocities
            X_norm_list.append(np.sqrt(csi_hat_norm))
        
            p_bar = [p1_hat_x, p1_hat_y, p2_hat_x, p2_hat_y, p3_hat_x, p3_hat_y, p4_hat_x, p4_hat_y] # .T
            v_bar = [v1_hat_x, v1_hat_y, v2_hat_x, v2_hat_y, v3_hat_x, v3_hat_y, v4_hat_x, v4_hat_y] # .T
    
            # [p_bar^T, v_bar^T]^T
            X_bar = np.transpose(np.hstack((np.array(p_bar), np.array(v_bar)))).reshape(16,1)   # shape = (16,1)
            X_bar_T = np.transpose(X_bar).reshape(1,16)   # shape = (1,16)
    
            # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ Neural
            # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ Network
        
            H_t, P_t = PTs_model(X_bar_T)
            # P_w = np.loadtxt('/Users/ekeulseuji/Downloads/FinalP.txt')
            # P_t = torch.tensor(P_w, dtype=torch.float)
            # X_t = torch.tensor(X_bar_T, dtype=torch.float)
            # H_t = 0.5 * (X_t @ P_t @ X_t.T).squeeze()
            H_list.append(H_t) # ------------------------------------------- H(~all t)

            phi_val = phi_calculator(k3, t0, tp, t)
            dot_phi_val = dot_phi_calculator(k3, t0, tp, t)
        
        
            if round(t,5)<=round(t0+tp,5):
                U_t = (phi_val**(-k3))*H_t  
                U_list.append(U_t) # ------------------- U(~tp)
            
            if round(t,5)>=round(t0+h,5):
                # q1 positive definite (for all t) ============================================================
                Bar_A = torch.tensor(A_bar, dtype=torch.float)
                Bar_C = torch.tensor(C_bar, dtype=torch.float)
                Bar_A_T = torch.tensor(np.transpose(A_bar), dtype=torch.float)
                Bar_C_T = torch.tensor(np.transpose(C_bar), dtype=torch.float)
            
                PAC = torch.matmul(P_t, (Bar_A+Bar_C)) + torch.matmul((Bar_A_T+Bar_C_T), P_t) + 2*kappa*P_t

                eigvals_PAC = torch.linalg.eigvals(-PAC).real
                max_eig_PAC = torch.max(eigvals_PAC)
            
                eigvals_P_inv = torch.linalg.eigvals(torch.linalg.inv(P_t)).real
                max_eig_P_inv = torch.max(eigvals_P_inv)
            
                q1 = max_eig_PAC * max_eig_P_inv
                q1_list.append(q1)

                loss1 = torch.max(zero_tensor, -q1) # q1 > 0
                loss1_sum = loss1_sum + loss1
                
                # R(t) ≥ 0 (for t ∈ [t0, t0+tp)) ====================================================================
                if round(t,5)>=round(t0+h,5) and round(t,5)<round(t0+tp,5):
                    cD_U = Caputo_frac_derivative_calculator(t, t0, h, U_list, q, alpha, beta, c, b, kappa, k1, k2, k3, tp)
                    cD_H = Caputo_frac_derivative_calculator(t, t0, h, H_list, q, alpha, beta, c, b, kappa, k1, k2, k3, tp)

                    phi_k = phi_val**(-k3)
                    phi_k_minus1 = phi_val**(-k3-1)

                    R_t = (phi_k*cD_H - k3*phi_k_minus1*dot_phi_val*H_t - cD_U)
                
                    R_list.append(R_t)
                    lossR = torch.max(zero_tensor, (-R_t))
                    loss2 = torch.max(zero_tensor, (-R_t)) # R(t) ≥ 0
                else:
                    lossR = torch.tensor(0, dtype=torch.float)
                    loss2 = torch.tensor(0, dtype=torch.float)

                lossR_sum = lossR_sum + lossR
                loss2_sum = loss2_sum + loss2*1000

                running_loss = running_loss + loss1 + loss2*1000
            
        # all eigenvalues should be greater than 0 (for positive definite square matrix)
        min_eig_P = torch.min(torch.linalg.eigvals(P_t).real)

        model_loss = torch.max(zero_tensor, torch.abs(H_t)-H_epsilon) + torch.max(zero_tensor,-min_eig_P) + running_loss/((tf-t0)/h)
        
        if model_loss.item()<=0.0:
            print('============================== SIMULATION DONE ==============================')
            # np.savetxt('/Users/ekeulseuji/Downloads/FINAL_P.txt', P_t.detach().numpy(), fmt='%.4f', delimiter=' ')
            print('At Iteration', n_iter)
            print('Final min_eig_P = ', min_eig_P)
            print('\n')
            break;
        
        elif model_loss.item()>0.0:
            optimizer.zero_grad()
            # backward propagation
            model_loss.backward()
    
            optimizer.step()
    
            # check: P_mat is positive definite or not for t∈[t0,tf)
    
            # save and print the loss
            losses.append(model_loss.item())
        
        n_iter = n_iter + 1

    print('The', i, 'th Sensitivity Analysis Simulation Series')
    tp_error = H_list[round((tp-t0)/h)]
    print("The bar_Xi Loss:", tp_error)

    total_R_loss = lossR_sum.detach().numpy()    
    print("Total R Loss:", total_R_loss)
        
    return tp_error.detach().numpy(), total_R_loss, H_list, q1_list, R_list, P, V, t_list, tp

## **SA Main - Random Params and Random Initial State for Each Anal**

In [22]:
res_dict = []

In [23]:
number_of_sa_tests = 20

for _ in range(0, number_of_sa_tests):
    num = random.randint(1, 100)
    print(num)
    
    np.random.seed(num)

    # randomly draw 1 of the combinations out, and perform the Main Algorithm
    n_samples = 1
    random_indices = np.random.choice(len(param_values), size=n_samples, replace=False)
    sampled_params = param_values[random_indices]

    print('q:', sampled_params[0][0], '\n',
      'alpha:', sampled_params[0][1], '\n',
      'beta:', sampled_params[0][2], '\n',
      'c:', sampled_params[0][3], '\n',
      'b:', sampled_params[0][4], '\n',
      'kappa:', sampled_params[0][5], '\n',
      'k1:', sampled_params[0][6], '\n',
      'k2:', sampled_params[0][7], '\n',
      'k3:', sampled_params[0][8], '\n',
      'tp:', sampled_params[0][9])
    
    # set for every PV value at different time points, for plotting and also for calculating Pc
    P = {'p0x':[], 'p0y':[], 'p1x':[], 'p1y':[], 'p2x':[], 'p2y':[], 'p3x':[], 'p3y':[], 'p4x':[], 'p4y':[]}
    V = {'v0x':[], 'v0y':[], 'v1x':[], 'v1y':[], 'v2x':[], 'v2y':[], 'v3x':[], 'v3y':[], 'v4x':[], 'v4y':[]}

    # number of input points, i.e. length of p0 = 50
    input_size = 1

    # order 4 graph (according to Figure 1(b), directed blue arrows)
    Vertex = {'1x', '1y', '2x', '2y', '3x', '3y', '4x', '4y'}

    # set p0 manually, create pi(p1, p2, ..., pn) randomly
    def create_p(size):
        p = []
        for s in range(size):
            p_elem = np.random.uniform(0,15)
            p.append(round(p_elem,3))
        return p

    p0 = np.array([[7.5], [7.5]])
    pi = []

    # follow the topological structure of multi-UAV systems described in figure 1??

    for i in range(n):
        pi.append(np.array([create_p(input_size), create_p(input_size)]))

    
    # create v0, vi(v1, v2, ..., vn)
    def create_v(size): # D^gamma_t p0(t) = v0(t)? constraint?
        v = []
        for s in range(size):
            v_elem = np.random.uniform(-5,5)
            v.append(round(v_elem,3))
        return v

    v0 = np.array([[2.5], [2.5]])
    vi = []

    for i in range(n):
        vi.append(np.array([create_v(input_size), create_v(input_size)]))


    input_pv = []
    for i in range(input_size):
        for j in range(n):
            input_elem_x = [p0[0][i], v0[0][i], pi[j][0][i], vi[j][0][i]]
            input_elem_y = [p0[1][i], v0[1][i], pi[j][1][i], vi[j][1][i]]
            input_pv.append(input_elem_x)
            input_pv.append(input_elem_y)


    P['p0x'].append(input_pv[0][0]); V['v0x'].append(input_pv[0][1])
    P['p0y'].append(input_pv[1][0]); V['v0y'].append(input_pv[1][1])
    P['p1x'].append(input_pv[0][2]); V['v1x'].append(input_pv[0][3])
    P['p1y'].append(input_pv[1][2]); V['v1y'].append(input_pv[1][3])
    P['p2x'].append(input_pv[2][2]); V['v2x'].append(input_pv[2][3])
    P['p2y'].append(input_pv[3][2]); V['v2y'].append(input_pv[3][3])
    P['p3x'].append(input_pv[4][2]); V['v3x'].append(input_pv[4][3])
    P['p3y'].append(input_pv[5][2]); V['v3y'].append(input_pv[5][3])
    P['p4x'].append(input_pv[6][2]); V['v4x'].append(input_pv[6][3])
    P['p4y'].append(input_pv[7][2]); V['v4y'].append(input_pv[7][3])

    Htp_err, R_loss, H_list, q1_list, R_list, P, V, t_list, tp = run_SA_simulation(sampled_params[0], P, V)
    print('Htp_error:', Htp_err)
    print('R_loss:', R_loss)

    plot_n = round((tf-t0)/h)
    t_plot = t_list

    p0_plot = []
    p1_plot = []
    p2_plot = []
    p3_plot = []
    p4_plot = []

    plt.rcParams['font.family'] = 'Times New Roman'

    # plot the starting n points?
    for i in range(plot_n):
        p0_plot.append([t_plot[i], P['p0x'][1:][i], P['p0y'][1:][i]])
        p1_plot.append([t_plot[i], P['p1x'][1:][i], P['p1y'][1:][i]])
        p2_plot.append([t_plot[i], P['p2x'][1:][i], P['p2y'][1:][i]])
        p3_plot.append([t_plot[i], P['p3x'][1:][i], P['p3y'][1:][i]])
        p4_plot.append([t_plot[i], P['p4x'][1:][i], P['p4y'][1:][i]])

    fig = plt.figure(figsize=(10,7.5))
    ax = fig.add_subplot(111, projection='3d')


    for elem in p0_plot[:round((tp-t0)/h)-2]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='firebrick', linewidths=1, alpha=0.5)

    for elem in p1_plot[:round((tp-t0)/h)-2]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#A8A8A8', linewidths=1, alpha=0.7)

    for elem in p2_plot[:round((tp-t0)/h)-2]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#8C8C8C', linewidths=1, alpha=0.7)

    for elem in p3_plot[:round((tp-t0)/h)-2]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#696969', linewidths=1, alpha=0.7)

    for elem in p4_plot[:round((tp-t0)/h)-2]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#363636', linewidths=1, alpha=0.7)

    
    x_grid = np.linspace(0, 13, 11)
    z_grid = np.linspace(0, 13, 11)
    xa, za = np.meshgrid(x_grid, z_grid)

    ta = np.full_like(xa, tp+0.015) 

    ax.plot_surface(xa, ta, za, color='m', alpha=0.05, edgecolor='m', linewidth=0.5)


    for elem in p0_plot[round((tp-t0)/h)-1:round((tp-t0)/h)]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='firebrick', linewidths=1, alpha=0.5)

    for elem in p1_plot[round((tp-t0)/h)-1:round((tp-t0)/h)]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='m', linewidths=1, alpha=0.7)

    for elem in p2_plot[round((tp-t0)/h)-1:round((tp-t0)/h)]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='m', linewidths=1, alpha=0.7)

    for elem in p3_plot[round((tp-t0)/h)-1:round((tp-t0)/h)]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='m', linewidths=1, alpha=0.7)

    for elem in p4_plot[round((tp-t0)/h)-1:round((tp-t0)/h)]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='m', linewidths=1, alpha=0.7)

    
    for elem in p0_plot[round((tp-t0)/h):]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='firebrick', linewidths=1, alpha=0.5)

    for elem in p1_plot[round((tp-t0)/h):]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#A8A8A8', linewidths=1, alpha=0.7)

    for elem in p2_plot[round((tp-t0)/h):]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#8C8C8C', linewidths=1, alpha=0.7)

    for elem in p3_plot[round((tp-t0)/h):]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#696969', linewidths=1, alpha=0.7)

    for elem in p4_plot[round((tp-t0)/h):]:
        ax.scatter(elem[1], elem[0], elem[2], s=200, c=elem[1], cmap='Greys', edgecolor='#363636', linewidths=1, alpha=0.7)

    ax.scatter(P['p1x'][-1], t_plot[-1], P['p1y'][-1], s=200, c=P['p1x'][-1], cmap='Greys', edgecolor='#A8A8A8', linewidths=1, label='$p_1$', alpha=0.7)
    # ax.annotate('p1', (P['p1x'][plot_n], P['p1y'][plot_n]))  

    ax.scatter(P['p2x'][-1], t_plot[-1], P['p2y'][-1], s=200, c=P['p2x'][-1], cmap='Greys', edgecolor='#8C8C8C', linewidths=1, label='$p_2$', alpha=0.7)
    # ax.annotate('p2', (P['p2x'][0], P['p2y'][0]))  

    ax.scatter(P['p3x'][-1], t_plot[-1], P['p3y'][-1], s=200, c=P['p3x'][-1], cmap='Greys', edgecolor='#696969', linewidths=1, label='$p_3$', alpha=0.7)
    # ax.annotate('p3', (P['p3x'][0], P['p3y'][0]))  

    ax.scatter(P['p4x'][-1], t_plot[-1], P['p4y'][-1], s=200, c=P['p4x'][-1], cmap='Greys',  edgecolor='#363636', linewidths=1, label='$p_4$', alpha=0.7)
    # ax.annotate('p4', (P['p4x'][0], P['p4y'][0]))  

    ax.scatter(P['p0x'][-1], t_plot[-1], P['p0y'][-1], s=200, c=P['p0x'][-1], cmap='Greys', edgecolor='firebrick', linewidths=1, label='$p_0$', alpha=0.5)
    # ax.annotate('p0', (P['p0x'][plot_n], P['p0y'][plot_n]))  

    ax.set_xlabel('Position in the X-axis', fontname='Times New Roman', fontsize=14)
    ax.set_ylabel('Time t', fontname='Times New Roman', fontsize=14)
    ax.set_zlabel('Position in the Y-axis', fontname='Times New Roman', fontsize=14)

    ax.set_title('The Controlled Multi-UAV System Dynamics', fontname='Times New Roman', fontsize=18)
    ax.view_init(elev=25.5, azim=-220.5)

    plt.grid()
    plt.legend()
    plt.savefig(f'/Users/ekeulseuji/Downloads/revPTS_3dDyn_{i+1}.pdf')
    plt.show()

    res_dict.append({'Random Seed': num, 'Htp_err': Htp_err, 'R_loss': R_loss, 'H_list': H_list, 'q1_list': q1_list, 'R_list': R_list, 'P': P, 'V': V})

41
q: 0.859765625 
 alpha: 1.30078125 
 beta: -1.76953125 
 c: 8.017578125 
 b: 9.912109375 
 kappa: 0.07607421874999999 
 k1: 2.8359375 
 k2: 7.900390625 
 k3: 1.0254701171875 
 tp: 0.1053125


KeyboardInterrupt: 

## **Analysis**

In [ ]:
'''
Y_errorX = []
Y_lossR = []

for param_set in param_values:
    err = run_simulation1_tpXi(param_set)
    loss = run_simulation1_tpR(param_set)
    Y_errorX.append(err)
    Y_lossR.append(loss)

Si1 = sobol.analyze(problem, np.array(Y_errorX))
Si2 = sobol.analyze(problem, np.array(Y_lossR))
'''

In [ ]:
Y_errorX 

In [ ]:
Y_lossR